In [724]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from harris import get_harris_corners

In [742]:
# tags: cepsr, north, northwestlib, pupin, street
img_tag = 'northwestlib'
impath1 = f'data/{img_tag}1.jpg'
impath2 = f'data/{img_tag}2.jpg'

img1 = cv2.imread(impath1)
img2 = cv2.imread(impath2)

In [743]:
def compress_image(img, percent=50):
    scale = percent / 100.0
    new_width = int(img.shape[1] * scale)
    new_height = int(img.shape[0] * scale)
    compressed_img = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_AREA)
    return compressed_img

# compressed = compress_image(img1, percent=60)
# save_image(compressed, impath1.split(".")[0])

# Step 1: Harris Corner Detection

In [ ]:
# Original images
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original Image 1")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
axes[1].set_title("Original Image 2")
axes[1].axis("off")

plt.tight_layout()
plt.savefig(f'out/{img_tag}_original_images.png')
plt.show()

In [745]:
# Convert to grayscale
gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0

# Harris corner detection
h1, coords1 = get_harris_corners(gray1, edge_discard=20)  # coords: 2 x N (y,x)
h2, coords2 = get_harris_corners(gray2, edge_discard=20)

print("Image 1 corners:", coords1.shape[1])
print("Image 2 corners:", coords2.shape[1])

Image 1 corners: 5345
Image 2 corners: 5295


In [ ]:
# Harris corners visualization
img1_draw = img1.copy()
img2_draw = img2.copy()

# coords is (2, N) -> y = coords[0,:], x = coords[1,:]
for i in range(coords1.shape[1]):
    y = int(coords1[0, i])
    x = int(coords1[1, i])
    cv2.circle(img1_draw, (x, y), 2, (0, 0, 255), 1)

for i in range(coords2.shape[1]):
    y = int(coords2[0, i])
    x = int(coords2[1, i])
    cv2.circle(img2_draw, (x, y), 2, (0, 0, 255), 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(cv2.cvtColor(img1_draw, cv2.COLOR_BGR2RGB))
axes[0].set_title("Image 1 with Harris Corners\n#points: {}".format(coords1.shape[1]))
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(img2_draw, cv2.COLOR_BGR2RGB))
axes[1].set_title("Image 2 with Harris Corners\n#points: {}".format(coords2.shape[1]))
axes[1].axis("off")

plt.tight_layout()
plt.savefig(f'out/{img_tag}_harris_corners.png')
plt.show()

# Step 2: Non-Maximal Suppression (NMS)

In [747]:
def nms(h, coords, window_size=11):
    r = window_size // 2

    height, width = h.shape
    ys = coords[0, :]
    xs = coords[1, :]

    coords_nms = []
    for y, x in zip(ys, xs):
        y = int(y)
        x = int(x)

        # window bounds
        y0 = max(0, y - r)
        y1 = min(height, y + r + 1)
        x0 = max(0, x - r)
        x1 = min(width, x + r + 1)

        patch = h[y0:y1, x0:x1]
        if h[y, x] == np.max(patch):
            coords_nms.append((y, x))

    coords_nms = np.array(coords_nms)
    if len(coords_nms) == 0:
        return np.zeros((2, 0), dtype=int)

    return coords_nms.T

In [759]:
window_size = 35
coords1_nms = nms(h1, coords1, window_size=window_size)
coords2_nms = nms(h2, coords2, window_size=window_size)

print('Before NMS:')
print('Image 1:', coords1.shape[1], 'corners')
print('Image 2:', coords2.shape[1], 'corners')

print('After NMS:')
print('Image 1:', coords1_nms.shape[1], 'corners')
print('Image 2:', coords2_nms.shape[1], 'corners')

Before NMS:
Image 1: 5345 corners
Image 2: 5295 corners
After NMS:
Image 1: 129 corners
Image 2: 128 corners


In [ ]:
# Harris corners visualization after NMS
img1_draw = img1.copy()
img2_draw = img2.copy()

for i in range(coords1_nms.shape[1]):
    y = int(coords1_nms[0, i])
    x = int(coords1_nms[1, i])
    cv2.circle(img1_draw, (x, y), 2, (0, 0, 255), 1)

for i in range(coords2_nms.shape[1]):
    y = int(coords2_nms[0, i])
    x = int(coords2_nms[1, i])
    cv2.circle(img2_draw, (x, y), 2, (0, 0, 255), 1)


fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(cv2.cvtColor(img1_draw, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Image 1 with NMS Harris Corners (window size: {window_size})\n#points: {coords1_nms.shape[1]}')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img2_draw, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Image 2 with NMS Harris Corners (window size: {window_size})\n#points: {coords2_nms.shape[1]}')
axes[1].axis('off')

plt.tight_layout()
plt.savefig(f'out/{img_tag}_nms_harris_corners.png')
plt.show()

# Step 3: Feature Descriptor Extraction

In [761]:
from skimage.transform import resize

def extract_descriptors(img, coords, patch_size=40, desc_size=8):
    r = patch_size // 2
    height, width = img.shape

    descriptor_patches = []
    descriptors = []
    valid_coords = []
    for i in range(coords.shape[1]):
        y = int(coords[0, i])
        x = int(coords[1, i])

        # check 40x40 patch fits in image
        if y - r < 0 or y + r >= height or x - r < 0 or x + r >= width:
            continue

        patch = img[y - r:y + r, x - r:x + r]  # 40x40

        # downsample to 8x8 with anti-aliasing
        patch_downsampled = resize(patch, (desc_size, desc_size), anti_aliasing=True)
        descriptor_patches.append(patch_downsampled)

        # flatten
        patch_flat = patch_downsampled.flatten()

        # bias/gain normalize (zero mean, unit std)
        patch_nobias = patch_flat - np.mean(patch_flat)
        std = np.std(patch_flat)
        patch_normalized = patch_nobias / (std + 1e-10)

        descriptors.append(patch_normalized)
        valid_coords.append((y, x))

    descriptors = np.array(descriptors)  # (N_valid, 64)

    if len(valid_coords) == 0:
        valid_coords = np.zeros((2, 0), dtype=int)
    else:
        valid_coords = np.array(valid_coords).T  # 2 x N_valid

    return descriptor_patches, descriptors, valid_coords

In [762]:
descriptor_patches1, descriptors1, coords1_valid = extract_descriptors(gray1, coords1_nms)
descriptor_patches2, descriptors2, coords2_valid = extract_descriptors(gray2, coords2_nms)

print('Descriptors image 1:', descriptors1.shape)  # (N1, 64)
print('Descriptors image 2:', descriptors2.shape)  # (N2, 64)

Descriptors image 1: (129, 64)
Descriptors image 2: (128, 64)


# Step 4: Feature Matching

In [763]:
def filter_matches(matches, thresh):
    good_matches = [m for m in matches if m[5] < thresh]
    good_matches.sort(key=lambda x: x[5])
    return good_matches

def compute_ssd(descriptors1, descriptors2):
    # (a - b)^2 = a^2 + b^2 - 2ab
    a2 = np.sum(descriptors1**2, axis=1, keepdims=True)      # (N1, 1)
    b2 = np.sum(descriptors2**2, axis=1, keepdims=True).T    # (1, N2)
    ab = descriptors1 @ descriptors2.T                       # (N1, N2)
    dists = a2 + b2 - 2 * ab
    return dists

def nndr_matches(descriptors1, descriptors2):
    """
    Returns:
        matches: list of tuples (i, j1, j2, d1, d2, ratio)
            i = index in descriptors1
            j1 = best match in descriptors2
            j2 = second best in descriptors2
    """
    dists = compute_ssd(descriptors1, descriptors2)

    matches = []
    for i in range(dists.shape[0]):
        row = dists[i]

        # indices of sorted distances
        nn = np.argsort(row)
        j1, j2 = nn[0], nn[1]

        d1 = row[j1]
        d2 = row[j2]

        ratio = d1 / (d2 + 1e-10)

        matches.append((i, j1, j2, d1, d2, ratio))

    return matches

In [764]:
def plot_nndr_hist(matches, thresh=0.7, bins=50):
    ratios = np.array([m[5] for m in matches])

    plt.figure(figsize=(8, 5))
    plt.hist(ratios, bins=bins, edgecolor='black')
    plt.axvline(thresh, linewidth=2, color='r', linestyle='dashed')
    plt.legend(['Threshold = {:.2f}'.format(thresh)])
    plt.title(f"NNDR Histogram (threshold = {thresh}, similarity metric: SSD)")
    plt.xlabel("NNDR = d(1NN) / d(2NN)")
    plt.ylabel("Frequency")
    plt.savefig(f'out/{img_tag}_nndr_hist.png')
    plt.show()

def show_top5_matches(descriptor_patches1, descriptor_patches2, matches_good, top_k=5):
    k = min(top_k, len(matches_good))
    if k == 0:
        print("No matches passed the threshold.")
        return

    fig, axes = plt.subplots(k, 3, figsize=(8, 2*k))

    if k == 1:
        axes = np.array([axes])

    for row in range(k):
        i, j1, j2, d1, d2, ratio = matches_good[row]

        p1 = descriptor_patches1[i]
        p2a = descriptor_patches2[j1]
        p2b = descriptor_patches2[j2]

        # Show patches
        axes[row, 0].imshow(p1, cmap="gray")
        axes[row, 1].imshow(p2a, cmap="gray")
        axes[row, 2].imshow(p2b, cmap="gray")

        # Titles
        axes[row, 0].set_title(f"NNDR = {ratio:.3f}\nImage 1 - Feature #{row+1}")
        axes[row, 1].set_title("Image 2 - 1NN")
        axes[row, 2].set_title("Image 2 - 2NN")

        # Turn off axes
        for col in range(3):
            axes[row, col].axis("off")

    fig.suptitle(
        f"Top {k} Best Matches by NNDR\n"
        "Image 1 - Feature | Image 2 - 1NN | Image 2 - 2NN",
        fontsize=14
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'out/{img_tag}_top5_patches.png')
    plt.show()

In [765]:
def draw_matches(img1, img2, coords1, coords2, matches_good, thresh, max_draw=None):
    H1, W1 = img1.shape[:2]
    H2, W2 = img2.shape[:2]

    canvas_h = max(H1, H2)
    canvas_w = W1 + W2

    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
    canvas[:H1, :W1] = img1
    canvas[:H2, W1:W1+W2] = img2

    # Compute statistics over all matches
    all_matched1 = set(i for i, _, _, _, _, _ in matches_good)
    all_matched2 = set(j1 for _, j1, _, _, _, _ in matches_good)

    total1 = coords1.shape[1]
    total2 = coords2.shape[1]

    matched_count1 = len(all_matched1)
    matched_count2 = len(all_matched2)
    unmatched1 = total1 - matched_count1
    unmatched2 = total2 - matched_count2

    # Draw matches
    matches_to_draw = matches_good[:max_draw] if max_draw is not None else matches_good
    radius = 3

    for match_idx, (i, j1, j2, d1, d2, ratio) in enumerate(matches_to_draw):
        y1, x1 = int(coords1[0, i]), int(coords1[1, i])
        y2, x2 = int(coords2[0, j1]), int(coords2[1, j1])

        # green line
        cv2.line(canvas, (x1, y1), (x2 + W1, y2), (0, 255, 0), 1)

        # circles
        cv2.circle(canvas, (x1, y1), radius, (0, 255, 0), -1)
        cv2.circle(canvas, (x1, y1), radius, (255, 255, 255), 1)
        cv2.circle(canvas, (x2 + W1, y2), radius, (0, 255, 0), -1)
        cv2.circle(canvas, (x2 + W1, y2), radius, (255, 255, 255), 1)

        # index labels
        cv2.putText(canvas, str(match_idx), (x1 + 4, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
        cv2.putText(canvas, str(match_idx), (x2 + W1 + 4, y2 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

    # Draw unmatched corners in red
    radius_unmatched = 2
    for i in range(total1):
        if i not in all_matched1:
            y, x = int(coords1[0, i]), int(coords1[1, i])
            cv2.circle(canvas, (x, y), radius_unmatched, (0, 0, 255), -1)
            cv2.circle(canvas, (x, y), radius_unmatched, (255, 255, 255), 1)

    for j in range(total2):
        if j not in all_matched2:
            y, x = int(coords2[0, j]), int(coords2[1, j])
            cv2.circle(canvas, (x + W1, y), radius_unmatched, (0, 0, 255), -1)
            cv2.circle(canvas, (x + W1, y), radius_unmatched, (255, 255, 255), 1)

    # Plot
    fig = plt.figure(figsize=(14, 7))
    plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    plt.axis("off")

    # Top title
    if thresh is not None:
        fig.text(
            0.5, 0.99,
            f"Matches (NNDR < {thresh})",
            ha="center", va="top", fontsize=13
        )
        
    fig.text(
        0.25, 0.9,
        f"Image 1 - After Feature Matching\n"
        f"Matched: {matched_count1} | Unmatched: {unmatched1} | Total: {total1}",
        ha="center", va="top", fontsize=12
    )

    fig.text(
        0.75, 0.9,
        f"Image 2 - After Feature Matching\n"
        f"Matched: {matched_count2} | Unmatched: {unmatched2} | Total: {total2}",
        ha="center", va="top", fontsize=12
    )
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'out/{img_tag}_matches_line.png')
    plt.show()

In [766]:
def draw_color_coded_matches(img1, img2, coords1, coords2, matches_good, thresh=None, max_draw=None):
    from matplotlib import cm

    H1, W1 = img1.shape[:2]
    H2, W2 = img2.shape[:2]
    canvas_h = max(H1, H2)
    canvas_w = W1 + W2

    # Combine images
    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
    canvas[:H1, :W1] = img1
    canvas[:H2, W1:W1+W2] = img2

    matches_to_draw = matches_good[:max_draw] if max_draw is not None else matches_good
    n_matches = len(matches_to_draw)

    if n_matches == 0:
        print("No good matches to draw.")
        return

    # Elegant color palette
    cmap = plt.colormaps['tab20']  # get the colormap
    colors_rgb = [tuple(int(c*255) for c in cmap(i/n_matches)[:3][::-1]) for i in range(n_matches)]

    radius = 6

    # Draw matched circles
    for idx, (i, j1, j2, d1, d2, ratio) in enumerate(matches_to_draw):
        color = colors_rgb[idx]

        # Image1
        y1, x1 = int(coords1[0, i]), int(coords1[1, i])
        cv2.circle(canvas, (x1, y1), radius, color, -1)
        cv2.circle(canvas, (x1, y1), radius, (255, 255, 255), 1)

        # Image2
        y2, x2 = int(coords2[0, j1]), int(coords2[1, j1])
        cv2.circle(canvas, (x2 + W1, y2), radius, color, -1)
        cv2.circle(canvas, (x2 + W1, y2), radius, (255, 255, 255), 1)

    # Compute stats from ALL matches, independent of max_draw
    matched1_all = set(i for i, _, _, _, _, _ in matches_good)
    matched2_all = set(j1 for _, j1, _, _, _, _ in matches_good)

    total1 = coords1.shape[1]
    total2 = coords2.shape[1]

    matched_count1 = len(matched1_all)
    matched_count2 = len(matched2_all)

    unmatched1 = total1 - matched_count1
    unmatched2 = total2 - matched_count2

    # Plot
    fig = plt.figure(figsize=(14, 7))
    plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    plt.axis("off")

    # Top title
    if thresh is not None:
        fig.text(
            0.5, 0.99,
            f"Matches (NNDR < {thresh})",
            ha="center", va="top", fontsize=13
        )

    # Left/Right image stats
    fig.text(
        0.25, 0.9,
        f"Image 1 - After Feature Matching\n"
        f"Matched: {matched_count1} | Unmatched: {unmatched1} | Total: {total1}",
        ha="center", va="top", fontsize=12
    )

    fig.text(
        0.75, 0.9,
        f"Image 2 - After Feature Matching\n"
        f"Matched: {matched_count2} | Unmatched: {unmatched2} | Total: {total2}",
        ha="center", va="top", fontsize=12
    )

    # Add match index labels in tiny rounded boxes
    for idx, (i, j1, j2, d1, d2, ratio) in enumerate(matches_to_draw):
        color = np.array(colors_rgb[idx][::-1])/255.0  # BGR -> RGB normalized for Matplotlib

        # Image1
        y1, x1 = coords1[0, i], coords1[1, i]
        plt.gca().text(x1, y1, str(idx), fontsize=8, color='white',
                       ha='center', va='center',
                       bbox=dict(facecolor=color, edgecolor='white', boxstyle='round,pad=0.2'))

        # Image2
        y2, x2 = coords2[0, j1], coords2[1, j1]
        plt.gca().text(x2 + W1, y2, str(idx), fontsize=8, color='white',
                       ha='center', va='center',
                       bbox=dict(facecolor=color, edgecolor='white', boxstyle='round,pad=0.2'))

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'out/{img_tag}_matches_colored.png')
    plt.show()

In [ ]:
# compute matches
matches = nndr_matches(descriptors1, descriptors2)

# NNDR histogram
nndr_threshold = 0.5
plot_nndr_hist(matches, thresh=nndr_threshold)

# filter good matches
good_matches = filter_matches(matches, thresh=nndr_threshold)
print("Good matches:", len(good_matches))

# top 5 matches
show_top5_matches(descriptor_patches1, descriptor_patches2, good_matches, top_k=5)

# visualize matches on the images
draw_matches(img1, img2, coords1_valid, coords2_valid, good_matches, nndr_threshold, max_draw=None)

# color-coded matches
draw_color_coded_matches(img1, img2, coords1_valid, coords2_valid, good_matches, nndr_threshold, max_draw=None)

# Panorama Stitching

In [768]:
from skimage.measure import ransac
from skimage.transform import ProjectiveTransform

def stitch_panorama(img1, img2, coords1, coords2, matches,
                    min_samples=4, residual_threshold=4.0, max_trials=2000):

    # extract matched coordinates of corresponding points
    pts1 = np.float32([[coords1[1, i], coords1[0, i]] for i, *_ in matches])
    pts2 = np.float32([[coords2[1, j], coords2[0, j]] for _, j, *_ in matches])

    # RANSAC homography
    model, inliers_mask = ransac(
        (pts1, pts2),
        ProjectiveTransform,
        min_samples=min_samples,
        residual_threshold=residual_threshold,
        max_trials=max_trials
    )

    homography_transform = model.params
    inliers = [match for match, is_inlier in zip(matches, inliers_mask) if is_inlier]

    # compute output panaroma bounds
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    corners1 = np.float32([[0,0],[w1,0],[w1,h1],[0,h1]])
    corners1_reshaped = corners1.reshape(1, -1, 2)   # shape (1, 4, 2)
    corners1_warped = cv2.perspectiveTransform(corners1_reshaped, homography_transform)[0]
    corners2 = np.float32([[0,0],[w2,0],[w2,h2],[0,h2]])

    all_corners = np.vstack((corners1_warped, corners2))
    x_min, y_min = np.floor(all_corners.min(axis=0)).astype(int)
    x_max, y_max = np.ceil(all_corners.max(axis=0)).astype(int)

    # translate to positive
    translation_transform = np.array(
        [[1, 0, -1 * x_min],
        [0, 1, -1 * y_min],
        [0, 0, 1]], dtype=np.float32)

    width, height = x_max - x_min, y_max - y_min

    # warp images
    warped1 = cv2.warpPerspective(img1, translation_transform @ homography_transform, (width, height))
    warped2 = cv2.warpPerspective(img2, translation_transform, (width, height))

    # blend by distance
    mask1 = (warped1.sum(axis = 2) > 0).astype(np.uint8)
    mask2 = (warped2.sum(axis = 2) > 0).astype(np.uint8)

    dist1 = cv2.distanceTransform(mask1, cv2.DIST_L2, 5)
    dist2 = cv2.distanceTransform(mask2, cv2.DIST_L2, 5)

    alpha = dist1 / (dist1 + dist2 + 1e-8)
    alpha_3ch = np.repeat(alpha[:, :, np.newaxis], 3, axis=2)
    panorama = (warped1 * alpha_3ch + warped2 * (1 - alpha_3ch)).clip(0, 255).astype(np.uint8)

    return panorama, inliers, inliers_mask, len(matches)

panorama, inliers, inlier_mask, total_matches = stitch_panorama(img1, img2, coords1_valid, coords2_valid, good_matches)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
ax.imshow(cv2.cvtColor(panorama, cv2.COLOR_BGR2RGB))
ax.axis('off')
ax.set_title(f'Panorama ({inlier_mask.sum()}/{total_matches} RANSAC inliers - {inlier_mask.sum() / total_matches * 100:.1f}%)', fontsize=14)
plt.tight_layout()
plt.savefig(f'out/{img_tag}_panorama.png', bbox_inches='tight')
plt.show()